<a href="https://colab.research.google.com/github/azrihasin/gaussian-splatting/blob/main/gaussian_splatting_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%cd /content
!git clone https://github.com/BaowenZ/RaDe-GS.git --recursive
!pip install -q plyfile

%cd /content/RaDe-GS
!pip install -q /content/RaDe-GS/submodules/diff-gaussian-rasterization
!pip install -q /content/RaDe-GS/submodules/simple-knn/

# Step 1: Install missing CUDA headers
!apt-get update && apt-get install -y nvidia-cuda-toolkit

# Step 3: Navigate to the project
%cd /content/RaDe-GS/submodules/tetra_triangulation

# Step 4: Clean and rebuild
!rm -rf CMakeFiles CMakeCache.txt  # Optional cleanup
!cmake . -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc
!make

# Step 5: Install
!pip install -e .

/content
fatal: destination path 'RaDe-GS' already exists and is not an empty directory.
/content/RaDe-GS
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for simple_knn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (simple_knn)
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.u

In [4]:
# Combined installation script for CUDA 12.1 + PyTorch + simple-knn

# 1. Install CUDA 12.1 and system dependencies
!wget https://developer.download.nvidia.com/compute/cuda/12.1.1/local_installers/cuda-repo-ubuntu2204-12-1-local_12.1.1-530.30.02-1_amd64.deb
!sudo dpkg -i cuda-repo-ubuntu2204-12-1-local_12.1.1-530.30.02-1_amd64.deb
!sudo cp /var/cuda-repo-ubuntu2204-12-1-local/cuda-*-keyring.gpg /usr/share/keyrings/
!sudo apt-get update
!sudo apt-get install -y cuda-toolkit-12-1 build-essential ninja-build

# 2. Set environment variables
import os
os.environ['PATH'] = '/usr/local/cuda-12.1/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.1/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'  # For T4 GPU

# 3. Install PyTorch for CUDA 12.1
!pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121

# 4. Clone repository (if needed)
!git clone https://github.com/yourusername/RaDe-GS.git /content/RaDe-GS
%cd /content/RaDe-GS
!git submodule update --init --recursive

# 5. Patch simple-knn code
!sed -i "1i #include <cfloat>" /content/RaDe-GS/submodules/simple-knn/simple_knn.cu

# 6. Install simple-knn
%cd /content/RaDe-GS/submodules/simple-knn
!pip install -v .

# 7. Verify installations
print("\n\033[1mVerification:\033[0m")
!nvcc --version
!nvidia-smi
import torch
print(f"\nPyTorch CUDA version: {torch.version.cuda}")
print(f"simple-knn CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

try:
    import simple_knn
    print("\n\033[92m✓ simple-knn installed successfully!\033[0m")
except ImportError:
    print("\n\033[91m✗ simple-knn installation failed!\033[0m")

--2025-02-08 05:18:54--  https://developer.download.nvidia.com/compute/cuda/12.1.1/local_installers/cuda-repo-ubuntu2204-12-1-local_12.1.1-530.30.02-1_amd64.deb
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.212.250.17, 23.212.250.16, 23.212.250.14
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.212.250.17|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3240527650 (3.0G) [application/x-deb]
Saving to: ‘cuda-repo-ubuntu2204-12-1-local_12.1.1-530.30.02-1_amd64.deb’

cuda-repo-ubuntu220 100%[===================>]   3.02G  53.0MB/s    in 78s     

2025-02-08 05:20:13 (39.5 MB/s) - ‘cuda-repo-ubuntu2204-12-1-local_12.1.1-530.30.02-1_amd64.deb’ saved [3240527650/3240527650]

Selecting previously unselected package cuda-repo-ubuntu2204-12-1-local.
(Reading database ... 129903 files and directories currently installed.)
Preparing to unpack cuda-repo-ubuntu2204-12-1-local_12.1.1-530.30.02-1_amd64.deb ...
Unp

In [5]:
%cd /content/RaDe-GS
!wget https://huggingface.co/datasets/azrihasin/test/resolve/main/tandt_db.zip
!unzip tandt_db.zip

/content/RaDe-GS
--2025-02-08 05:29:22--  https://huggingface.co/datasets/azrihasin/test/resolve/main/tandt_db.zip
Resolving huggingface.co (huggingface.co)... 3.171.171.128, 3.171.171.104, 3.171.171.6, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.128|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/58/7a/587ab5f03fc76cbcf107889b8716bed57b997f5bddb0271d7f5d2fdbd2693eb8/f8d08b30ed298b9fe53a9ac819719c8b5a3fdd58349cc9924ac4e9f8fc4888e3?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tandt_db.zip%3B+filename%3D%22tandt_db.zip%22%3B&response-content-type=application%2Fzip&Expires=1738996162&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczODk5NjE2Mn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzU4LzdhLzU4N2FiNWYwM2ZjNzZjYmNmMTA3ODg5Yjg3MTZiZWQ1N2I5OTdmNWJkZGIwMjcxZDdmNWQyZmRiZDI2OTNlYjgvZjhkMDhiMzBlZDI5OGI5ZmU1M2E5YWM4MTk3MTljOGI1YTNmZGQ1ODM0

In [6]:
%cd /content/RaDe-GS
!python train.py -s /content/RaDe-GS/tandt_db/tandt/showroom

/content/RaDe-GS
Traceback (most recent call last):
  File "/content/RaDe-GS/train.py", line 16, in <module>
    from gaussian_renderer import render, network_gui
  File "/content/RaDe-GS/gaussian_renderer/__init__.py", line 14, in <module>
    from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
  File "/usr/local/lib/python3.11/dist-packages/diff_gaussian_rasterization/__init__.py", line 15, in <module>
    from . import _C
ImportError: /usr/local/lib/python3.11/dist-packages/diff_gaussian_rasterization/_C.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN3c1021throwNullDataPtrErrorEv


In [1]:
nvcc --version

NameError: name 'nvcc' is not defined